# Phase E 

### Dataset

In [18]:
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (train_test_split,StratifiedKFold,cross_validate,GridSearchCV)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier,ExtraTreesClassifier,GradientBoostingClassifier)
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report,confusion_matrix,ConfusionMatrixDisplay)

def charger_reservations(chemin_csv):
    
    dataframe = pd.read_csv(chemin_csv)

   
    return dataframe
df = charger_reservations("Hotel Reservations.csv")

display(df.head())

,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,INN00001,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,2017,10,2,Offline,0,0,0,65.00,0,Not_Canceled
1,INN00002,2,0,2,3,Not Selected,0,Room_Type 1,5,2018,11,6,Online,0,0,0,106.68,1,Not_Canceled
2,INN00003,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,2018,2,28,Online,0,0,0,60.00,0,Canceled
3,INN00004,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,2018,5,20,Online,0,0,0,100.00,0,Canceled
4,INN00005,2,0,1,1,Not Selected,0,Room_Type 1,48,2018,4,11,Online,0,0,0,94.50,0,Canceled


### analyse du dataset

In [19]:
def analyser_reservations(dataframe):
   
    nombre_doublons = dataframe.duplicated().sum()
    nombre_valeurs_manquantes = dataframe.isna().sum().sum()

    repartition = pd.DataFrame({
        "Effectif": dataframe["booking_status"].value_counts(),
        "Pourcentage": (
            dataframe["booking_status"]
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
        )
    })

    print(f"Nombre de doublons : {nombre_doublons}")
    print(f"Nombre total de valeurs manquantes : {nombre_valeurs_manquantes}")
    print(
        f"Nombre d'identifiants uniques : "
        f"{dataframe['Booking_ID'].nunique()}"
    )

    return repartition
repartition_cible = analyser_reservations(df)

display(repartition_cible)

Nombre de doublons : 0
Nombre total de valeurs manquantes : 0
Nombre d'identifiants uniques : 36275


,Effectif,Pourcentage
booking_status,,
Not_Canceled,24390,67.24
Canceled,11885,32.76


### Prétraitement 

In [55]:
try:
    encodeur = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encodeur = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

preprocesseur = ColumnTransformer(
    transformers=[
        (
            "numerique",
            StandardScaler(),
            colonnes_numeriques
        ),
        (
            "categoriel",
            encodeur,
            colonnes_categorielles
        )
    ]
)



### Découpage train/test unique

In [82]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y)



### Définition des compétiteurs

In [58]:
competiteurs = {
    "LogisticRegression": LogisticRegression(
        max_iter=5000
    ),

    "DecisionTree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),

    "SVC_rbf": SVC(
        kernel="rbf",
        cache_size=2000
    )
}

print("Algorithmes participant au Fight :")

for nom_algorithme in competiteurs:
    print("-", nom_algorithme)

Algorithmes participant au Fight :
- LogisticRegression
- DecisionTree
- RandomForest
- GradientBoosting
- SVC_rbf


### Fonction Fight des IA

In [83]:
def fight_des_ia(
    X_train,
    X_test,
    y_train,
    y_test,
    metrique
):
   
    resultats = []

    for nom_algorithme, algorithme in competiteurs.items():

        print(f"Entraînement de {nom_algorithme}...")

        pipeline = Pipeline(
            steps=[
                (
                    "preprocesseur",
                    clone(preprocesseur)
                ),
                (
                    "modele",
                    clone(algorithme)
                )
            ]
        )

        debut = time.perf_counter()

        pipeline.fit(
            X_train,
            y_train
        )

        temps_entrainement = (
            time.perf_counter() - debut
        )

        predictions = pipeline.predict(
            X_test
        )

        score = metrique(
            y_test,
            predictions
        )

        resultats.append({
            "Algorithme": nom_algorithme,
            "F1_score": score,
            "Temps_entrainement_secondes": temps_entrainement
        })

    leaderboard = pd.DataFrame(
        resultats
    )

    leaderboard = leaderboard.sort_values(
        by="F1_score",
        ascending=False
    ).reset_index(drop=True)

    leaderboard.index = leaderboard.index + 1
    leaderboard.index.name = "Rang"

    return leaderboard
leaderboard = fight_des_ia(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    metrique=f1_score
)

print("\n LEADERBOARD HOTEL RESERVATIONS ")

display(
    leaderboard.style.format({
        "F1_score": "{:.4f}",
        "Temps_entrainement_secondes": "{:.4f}"
    })
)

Entraînement de LogisticRegression...
Entraînement de DecisionTree...
Entraînement de RandomForest...
Entraînement de GradientBoosting...
Entraînement de SVC_rbf...

 LEADERBOARD HOTEL RESERVATIONS 


,Algorithme,F1_score,Temps_entrainement_secondes
Rang,,,
1,RandomForest,0.8528,1.6903
2,DecisionTree,0.8076,0.1282
3,GradientBoosting,0.7679,3.6091
4,SVC_rbf,0.7545,15.9798
5,LogisticRegression,0.6943,0.1243


### Identification du champion

In [84]:
champion_hotel = leaderboard_hotel.iloc[0]
deuxieme_hotel = leaderboard_hotel.iloc[1]

ecart_f1 = (
    champion_hotel["F1"]
    - deuxieme_hotel["F1"]
)

rapport_temps = (
    champion_hotel["Temps_entrainement_s"]
    / deuxieme_hotel["Temps_entrainement_s"]
)

print("ARBITRAGE DU FIGHT ")

print(
    "\nChampion :",
    champion_hotel["Algorithme"]
)

print(
    f"F1 du champion : "
    f"{champion_hotel['F1']:.4f}"
)

print(
    f"Temps d'entraînement du champion : "
    f"{champion_hotel['Temps_entrainement_s']:.4f} s"
)

print(
    "\nDeuxième :",
    deuxieme_hotel["Algorithme"]
)

print(
    f"F1 du deuxième : "
    f"{deuxieme_hotel['F1']:.4f}"
)

print(
    f"\nÉcart de F1 : {ecart_f1:.4f}"
)

print(
    f"Rapport entre les temps d'entraînement : "
    f"{rapport_temps:.2f}"
)

ARBITRAGE DU FIGHT 

Champion : RandomForest
F1 du champion : 0.8528
Temps d'entraînement du champion : 1.4057 s

Deuxième : DecisionTree
F1 du deuxième : 0.8076

Écart de F1 : 0.0452
Rapport entre les temps d'entraînement : 12.01


### Changement de métrique

#### Fonction de comparaison

In [85]:
def comparer_metriques(leaderboard):
    
    resultats = []

    for metrique in [
        "Accuracy",
        "F1",
        "Recall"
    ]:
        classement = leaderboard.sort_values(
            by=metrique,
            ascending=False
        ).reset_index(drop=True)

        classement.index = classement.index + 1

        for rang, ligne in classement.iterrows():
            resultats.append({
                "Metrique": metrique,
                "Rang": rang,
                "Algorithme": ligne["Algorithme"],
                "Score": ligne[metrique]
            })

    return pd.DataFrame(resultats)
classements_metriques = comparer_metriques(
    leaderboard_hotel
)

display(
    classements_metriques.style.format({
        "Score": "{:.4f}"
    })
)

champions_metriques = (
    classements_metriques[
        classements_metriques["Rang"] == 1
    ]
    .reset_index(drop=True)
)

print("\n CHAMPION SELON CHAQUE MÉTRIQUE ")

display(
    champions_metriques.style.format({
        "Score": "{:.4f}"
    })
)

,Metrique,Rang,Algorithme,Score
0,Accuracy,1,RandomForest,0.9075
1,Accuracy,2,DecisionTree,0.8744
2,Accuracy,3,GradientBoosting,0.8584
3,Accuracy,4,SVC_rbf,0.8503
4,Accuracy,5,LogisticRegression,0.8143
5,F1,1,RandomForest,0.8528
6,F1,2,DecisionTree,0.8076
7,F1,3,GradientBoosting,0.7679
8,F1,4,SVC_rbf,0.7545
9,F1,5,LogisticRegression,0.6943



 CHAMPION SELON CHAQUE MÉTRIQUE 


,Metrique,Rang,Algorithme,Score
0,Accuracy,1,RandomForest,0.9075
1,F1,1,RandomForest,0.8528
2,Recall,1,RandomForest,0.8174


### Validation croisée

In [87]:
def valider_modeles(X_train,y_train,preprocesseur,competiteurs):
   
    validation = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    resultats = []

    for nom_algorithme, algorithme in competiteurs.items():

        print(
            f"Validation croisée de "
            f"{nom_algorithme}..."
        )

        pipeline = Pipeline(
            steps=[
                (
                    "preprocesseur",
                    clone(preprocesseur)
                ),
                (
                    "modele",
                    clone(algorithme)
                )
            ]
        )

        scores = cross_validate(
            estimator=pipeline,
            X=X_train,
            y=y_train,
            cv=validation,
            scoring={
                "accuracy": "accuracy",
                "f1": "f1",
                "recall": "recall"
            },
            n_jobs=-1,
            return_train_score=False
        )

        resultats.append({
            "Algorithme": nom_algorithme,
            "Accuracy_CV_moyenne": (
                scores["test_accuracy"].mean()
            ),
            "F1_CV_moyen": (
                scores["test_f1"].mean()
            ),
            "F1_CV_ecart_type": (
                scores["test_f1"].std()
            ),
            "Recall_CV_moyen": (
                scores["test_recall"].mean()
            ),
            "Temps_entrainement_CV_s": (
                scores["fit_time"].mean()
            ),
            "Temps_prediction_CV_s": (
                scores["score_time"].mean()
            )
        })

    resultats_cv = pd.DataFrame(
        resultats
    )

    resultats_cv = resultats_cv.sort_values(
        by="F1_CV_moyen",
        ascending=False
    ).reset_index(drop=True)

    return resultats_cv
resultats_cv_hotel = valider_modeles(
    X_train=X_train,
    y_train=y_train,
    preprocesseur=preprocesseur_hotel,
    competiteurs=competiteurs_hotel
)

print("\n VALIDATION CROISÉE ")

display(
    resultats_cv_hotel.style.format({
        "Accuracy_CV_moyenne": "{:.4f}",
        "F1_CV_moyen": "{:.4f}",
        "F1_CV_ecart_type": "{:.4f}",
        "Recall_CV_moyen": "{:.4f}",
        "Temps_entrainement_CV_s": "{:.4f}",
        "Temps_prediction_CV_s": "{:.4f}"
    })
)

Validation croisée de LogisticRegression...
Validation croisée de DecisionTree...
Validation croisée de RandomForest...
Validation croisée de GradientBoosting...
Validation croisée de SVC_rbf...

 VALIDATION CROISÉE 


,Algorithme,Accuracy_CV_moyenne,F1_CV_moyen,F1_CV_ecart_type,Recall_CV_moyen,Temps_entrainement_CV_s,Temps_prediction_CV_s
0,RandomForest,0.8960,0.8335,0.0052,0.7947,2.2340,0.1787
1,DecisionTree,0.8592,0.7866,0.0022,0.7924,0.0936,0.0233
2,GradientBoosting,0.8486,0.7511,0.0047,0.6976,1.9364,0.0259
3,SVC_rbf,0.8387,0.7315,0.0028,0.6707,6.9619,5.5353
4,LogisticRegression,0.8025,0.6745,0.0067,0.6246,0.0756,0.0124


### Optimisation du champion avec GridSearchCV

In [88]:
grilles_parametres = {"LogisticRegression": {"modele__C": [0.1,1,10]},

    "DecisionTree": {
        "modele__max_depth": [
            None,
            10,
            20
        ],
        "modele__min_samples_leaf": [
            1,
            2,
            5
        ]
    },

    "RandomForest": {
        "modele__n_estimators": [
            100,
            200,
            300
        ],
        "modele__max_depth": [
            None,
            20
        ],
        "modele__min_samples_leaf": [
            1,
            2
        ]
    },

    "GradientBoosting": {
        "modele__n_estimators": [
            100,
            200
        ],
        "modele__learning_rate": [
            0.05,
            0.1
        ]
    },

    "SVC_rbf": {
        "modele__C": [
            0.5,
            1,
            2
        ],
        "modele__gamma": [
            "scale",
            "auto"
        ]
    }
}

def optimiser_champion(
    leaderboard,
    X_train,
    X_test,
    y_train,
    y_test,
    preprocesseur,
    competiteurs,
    grilles_parametres
):
  
    nom_champion = leaderboard.iloc[0][
        "Algorithme"
    ]

    score_initial = leaderboard.iloc[0][
        "F1"
    ]

    modele_champion = clone(
        competiteurs[nom_champion]
    )

    if hasattr(
        modele_champion,
        "n_jobs"
    ):
        modele_champion.set_params(
            n_jobs=1
        )

    pipeline = Pipeline(
        steps=[
            (
                "preprocesseur",
                clone(preprocesseur)
            ),
            (
                "modele",
                modele_champion
            )
        ]
    )

    validation = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    recherche = GridSearchCV(
        estimator=pipeline,
        param_grid=grilles_parametres[
            nom_champion
        ],
        scoring="f1",
        cv=validation,
        n_jobs=-1,
        refit=True,
        verbose=1
    )

    debut_recherche = time.perf_counter()

    recherche.fit(
        X_train,
        y_train
    )

    temps_recherche = (
        time.perf_counter()
        - debut_recherche
    )

    meilleur_modele = (
        recherche.best_estimator_
    )

    debut_prediction = time.perf_counter()

    predictions = meilleur_modele.predict(
        X_test
    )

    temps_prediction = (
        time.perf_counter()
        - debut_prediction
    )

    score_optimise = f1_score(
        y_test,
        predictions
    )

    gain = (
        score_optimise
        - score_initial
    )

    comparaison = pd.DataFrame({
        "Version": [
            "Champion initial",
            "Champion optimisé"
        ],
        "F1": [
            score_initial,
            score_optimise
        ]
    })

    print("Champion optimisé :", nom_champion)

    

    print(
        "\nMeilleur F1 moyen en validation croisée :",
        round(recherche.best_score_, 4)
    )

    print(
        "Temps total de GridSearchCV :",
        round(temps_recherche, 4),
        "seconde(s)"
    )

    print(
        "Temps de prédiction du modèle optimisé :",
        round(temps_prediction, 4),
        "seconde(s)"
    )

    print(
        "Gain de F1 sur le test :",
        round(gain, 4)
    )

    return meilleur_modele, recherche, comparaison
(
    meilleur_modele,
    recherche_champion,
    comparaison_optimisation
) = optimiser_champion(
    leaderboard=leaderboard_hotel,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    preprocesseur=preprocesseur_hotel,
    competiteurs=competiteurs_hotel,
    grilles_parametres=grilles_parametres
)

print("\n COMPARAISON AVANT ET APRÈS GRIDSEARCHCV ")

display(
    comparaison_optimisation.style.format({
        "F1": "{:.4f}"
    })
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Champion optimisé : RandomForest

Meilleur F1 moyen en validation croisée : 0.8339
Temps total de GridSearchCV : 52.1712 seconde(s)
Temps de prédiction du modèle optimisé : 0.301 seconde(s)
Gain de F1 sur le test : -0.0007

 COMPARAISON AVANT ET APRÈS GRIDSEARCHCV 


,Version,F1
0,Champion initial,0.8528
1,Champion optimisé,0.8520


### Second Fight sur un autre dataset ( sonor ) 

### Fonction de chargement de Sonar

In [90]:
def charger_sonar(source):
    
    M = mine = 1
    R = rocher = 0
    
    sonar = pd.read_csv(
        source,
        header=None
    )

    X_sonar = sonar.iloc[
        :,
        :-1
    ].copy()

    y_sonar = sonar.iloc[
        :,
        -1
    ].map({
        "M": 1,
        "R": 0
    })

  

    print("\nRépartition de la cible :")
    print(
        y_sonar.value_counts()
        .rename({
            1: "Mines",
            0: "Rochers"
        })
    )

    return X_sonar, y_sonar
URL_SONAR = (
    "https://archive.ics.uci.edu/ml/"
    "machine-learning-databases/"
    "undocumented/connectionist-bench/"
    "sonar/sonar.all-data"
)

X_sonar, y_sonar = charger_sonar(
    URL_SONAR
)

display(X_sonar.head())


Répartition de la cible :
60
Mines      111
Rochers     97
Name: count, dtype: int64


,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0232,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0125,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0033,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0241,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0156,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094


### Split et prétraitement Sonar 

In [91]:
(X_train_sonar,X_test_sonar,y_train_sonar,y_test_sonar) = train_test_split(X_sonar,y_sonar,test_size=0.20,random_state=RANDOM_STATE,stratify=y_sonar)



### fight_des_ia

In [92]:
def fight_des_ia(
    X_train,
    X_test,
    y_train,
    y_test,
    preprocesseur,
    competiteurs,
    metrique,
    nom_metrique
):
    
    resultats = []

    for nom_algorithme, algorithme in competiteurs.items():

        print(f"Entraînement de {nom_algorithme}...")

        pipeline = Pipeline(
            steps=[
                (
                    "preprocesseur",
                    clone(preprocesseur)
                ),
                (
                    "modele",
                    clone(algorithme)
                )
            ]
        )

        debut_entrainement = time.perf_counter()

        pipeline.fit(
            X_train,
            y_train
        )

        temps_entrainement = (
            time.perf_counter()
            - debut_entrainement
        )

        debut_prediction = time.perf_counter()

        predictions = pipeline.predict(
            X_test
        )

        temps_prediction = (
            time.perf_counter()
            - debut_prediction
        )

        score_principal = metrique(
            y_test,
            predictions
        )

        resultats.append({
            "Algorithme": nom_algorithme,
            "Metrique": nom_metrique,
            "Score": score_principal,
            "Accuracy": accuracy_score(
                y_test,
                predictions
            ),
            "Precision": precision_score(
                y_test,
                predictions,
                zero_division=0
            ),
            "Recall": recall_score(
                y_test,
                predictions,
                zero_division=0
            ),
            "F1": f1_score(
                y_test,
                predictions,
                zero_division=0
            ),
            "Temps_entrainement_s": temps_entrainement,
            "Temps_prediction_s": temps_prediction
        })

    leaderboard = pd.DataFrame(
        resultats
    )

    leaderboard = leaderboard.sort_values(
        by=[
            "Score",
            "Temps_entrainement_s"
        ],
        ascending=[
            False,
            True
        ]
    ).reset_index(drop=True)

    leaderboard.index = leaderboard.index + 1
    leaderboard.index.name = "Rang"

    return leaderboard

### Compétiteurs Sonar 

In [95]:
competiteurs_sonar = {
    "LogisticRegression": LogisticRegression(
        max_iter=5000
    ),

    "DecisionTree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),

    "SVC_rbf": SVC(
        kernel="rbf"
    )
}
leaderboard_sonar = fight_des_ia(
    X_train=X_train_sonar,
    X_test=X_test_sonar,
    y_train=y_train_sonar,
    y_test=y_test_sonar,
    preprocesseur=preprocesseur_sonar,
    competiteurs=competiteurs_sonar,
    metrique=f1_score,
    nom_metrique="F1"
)

print("\n LEADERBOARD SONAR ")

display(
    leaderboard_sonar.style.format({
        "Score": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "Temps_entrainement_s": "{:.4f}",
        "Temps_prediction_s": "{:.4f}"
    })
)


Entraînement de LogisticRegression...
Entraînement de DecisionTree...
Entraînement de RandomForest...
Entraînement de GradientBoosting...
Entraînement de SVC_rbf...

 LEADERBOARD SONAR 


,Algorithme,Metrique,Score,Accuracy,Precision,Recall,F1,Temps_entrainement_s,Temps_prediction_s
Rang,,,,,,,,,
1,SVC_rbf,F1,0.9362,0.9286,0.8800,1.0000,0.9362,0.0026,0.0012
2,GradientBoosting,F1,0.8636,0.8571,0.8636,0.8636,0.8636,0.3323,0.0009
3,DecisionTree,F1,0.8571,0.8333,0.7778,0.9545,0.8571,0.0081,0.0012
4,LogisticRegression,F1,0.8444,0.8333,0.8261,0.8636,0.8444,0.0211,0.0018
5,RandomForest,F1,0.8444,0.8333,0.8261,0.8636,0.8444,0.2053,0.0258


### Fonction de comparaison des podiums

In [96]:
def comparer_podiums(
    leaderboard_hotel,
    leaderboard_sonar
):
    
    podium_hotel = (
        leaderboard_hotel
        .head(3)[
            [
                "Algorithme",
                "F1"
            ]
        ]
        .copy()
    )

    podium_hotel["Dataset"] = (
        "Hotel Reservations"
    )

    podium_hotel["Rang"] = [
        1,
        2,
        3
    ]

    podium_sonar = (
        leaderboard_sonar
        .head(3)[
            [
                "Algorithme",
                "F1"
            ]
        ]
        .copy()
    )

    podium_sonar["Dataset"] = "Sonar"

    podium_sonar["Rang"] = [
        1,
        2,
        3
    ]

    comparaison = pd.concat(
        [
            podium_hotel,
            podium_sonar
        ],
        ignore_index=True
    )

    comparaison = comparaison[
        [
            "Dataset",
            "Rang",
            "Algorithme",
            "F1"
        ]
    ]

    return comparaison
comparaison_podiums = comparer_podiums(
    leaderboard_hotel,
    leaderboard_sonar
)

display(
    comparaison_podiums.style.format({
        "F1": "{:.4f}"
    })
)

champion_hotel_nom = (
    leaderboard_hotel.iloc[0][
        "Algorithme"
    ]
)

champion_sonar_nom = (
    leaderboard_sonar.iloc[0][
        "Algorithme"
    ]
)

print(
    "\nChampion Hotel Reservations :",
    champion_hotel_nom
)

print(
    "Champion Sonar :",
    champion_sonar_nom
)

if champion_hotel_nom == champion_sonar_nom:
    print(
        "\nLe champion reste le même "
        "sur les deux datasets."
    )
else:
    print(
        "\nLe champion change selon le dataset."
    )

,Dataset,Rang,Algorithme,F1
0,Hotel Reservations,1,RandomForest,0.8528
1,Hotel Reservations,2,DecisionTree,0.8076
2,Hotel Reservations,3,GradientBoosting,0.7679
3,Sonar,1,SVC_rbf,0.9362
4,Sonar,2,GradientBoosting,0.8636
5,Sonar,3,DecisionTree,0.8571



Champion Hotel Reservations : RandomForest
Champion Sonar : SVC_rbf

Le champion change selon le dataset.
